In [ ]:
from google.colab import files

uploaded = files.upload()

Saving archive (17).zip to archive (17).zip


## Imports & configuration

In [ ]:
import os
import re
import shutil
import hashlib
import zipfile

import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve, classification_report,
)
import matplotlib.pyplot as plt

RANDOM_SEED = 42
IMG_SIZE = (224, 224)

## Extract dataset

In [ ]:

ZIP_PATH = "/content/archive (17).zip"
EXTRACT_DIR = "/content/brain_ct_extracted"
OUTPUT_DIR = "/content/patient_level_dataset"

MANIFEST_PATH = "/content/patient_level_split_manifest.csv"
CASE_MANIFEST_PATH = "/content/case_level_split_manifest.csv"

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("Dataset extracted successfully.")
print("Location:", EXTRACT_DIR)

image_files = [
    os.path.join(root, file)
    for root, _, files_in_dir in os.walk(EXTRACT_DIR)
    for file in files_in_dir
    if file.lower().endswith(IMAGE_EXTENSIONS)
]
print("Total image files found:", len(image_files))



Dataset extracted successfully.
Location: /content/brain_ct_extracted
Total image files found: 2515


## Parse class and patient (case) ID



In [ ]:

def extract_class(path):

    parts = os.path.normpath(path).split(os.sep)
    for part in parts:
        if part.lower() in ["normal", "stroke"]:
            return part.capitalize()
    return None


def extract_case_id(filename):

    match = re.match(r"^\s*(\d+)", filename)
    return match.group(1) if match else None


records = []
for path in image_files:
    filename = os.path.basename(path)
    class_name = extract_class(path)
    case_id = extract_case_id(filename)
    if class_name is None or case_id is None:
        continue
    records.append({
        "image_path": path, "filename": filename,
        "class": class_name, "case_id": case_id,
    })

df = pd.DataFrame(records)
print("Valid image records:", len(df))
print(df.groupby("class")["case_id"].nunique())



Valid image records: 2515
class
Normal    51
Stroke    31
Name: case_id, dtype: int64


In [ ]:

def calculate_sha256(filepath):
    sha256 = hashlib.sha256()
    with open(filepath, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            sha256.update(chunk)
    return sha256.hexdigest()


df["sha256"] = df["image_path"].apply(calculate_sha256)
df["duplicate_of"] = ""
first_occurrence = {}
for index, row in df.iterrows():
    h = row["sha256"]
    if h in first_occurrence:
        df.at[index, "duplicate_of"] = first_occurrence[h]
    else:
        first_occurrence[h] = row["image_path"]

df_unique = df[df["duplicate_of"] == ""].copy()
print("Unique images:", len(df_unique))
print("Unique patients:", df_unique[["class", "case_id"]].drop_duplicates().shape[0])



Unique images: 2501
Unique patients: 82


## Patient-level train / validation / test split



In [ ]:

case_df = df_unique[["class", "case_id"]].drop_duplicates().reset_index(drop=True)

train_val_cases, test_cases = train_test_split(
    case_df, test_size=17, stratify=case_df["class"], random_state=RANDOM_SEED,
)
train_cases, val_cases = train_test_split(
    train_val_cases, test_size=8, stratify=train_val_cases["class"], random_state=RANDOM_SEED,
)

print("Training patients   :", len(train_cases))
print("Validation patients :", len(val_cases))
print("Testing patients    :", len(test_cases))

train_case_set = set(zip(train_cases["class"], train_cases["case_id"]))
val_case_set = set(zip(val_cases["class"], val_cases["case_id"]))
test_case_set = set(zip(test_cases["class"], test_cases["case_id"]))


def assign_split(row):
    key = (row["class"], row["case_id"])
    if key in train_case_set:
        return "Train"
    if key in val_case_set:
        return "Validation"
    if key in test_case_set:
        return "Test"
    return None


df_unique["corrected_split"] = df_unique.apply(assign_split, axis=1)

train_ids = set(train_cases["case_id"])
val_ids = set(val_cases["case_id"])
test_ids = set(test_cases["case_id"])
assert not (train_ids & val_ids) and not (train_ids & test_ids) and not (val_ids & test_ids)
print("Patient overlap check passed — zero overlap between splits.")

train_df = df_unique[df_unique["corrected_split"] == "Train"].reset_index(drop=True)
val_df = df_unique[df_unique["corrected_split"] == "Validation"].reset_index(drop=True)
test_df = df_unique[df_unique["corrected_split"] == "Test"].reset_index(drop=True)
print("Train:", len(train_df), " Validation:", len(val_df), " Test:", len(test_df))


Training patients   : 57
Validation patients : 8
Testing patients    : 17
Patient overlap check passed — zero overlap between splits.
Train: 1761  Validation: 221  Test: 519


In [ ]:
for name, data in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    normal = (data["class"] == "Normal").sum()
    stroke = (data["class"] == "Stroke").sum()

    print(
        f"{name}: "
        f"Total = {len(data)}, "
        f"Normal = {normal}, "
        f"Stroke = {stroke}, "
        f"Patients = {data['case_id'].nunique()}"
    )

Train: Total = 1761, Normal = 1065, Stroke = 696, Patients = 57
Validation: Total = 221, Normal = 148, Stroke = 73, Patients = 8
Test: Total = 519, Normal = 338, Stroke = 181, Patients = 17


## Verify zero patient overlap between splits

In [ ]:
train_ids = set(train_cases["case_id"])
val_ids = set(val_cases["case_id"])
test_ids = set(test_cases["case_id"])

print("Train ∩ Validation:", train_ids & val_ids)
print("Train ∩ Test:      ", train_ids & test_ids)
print("Validation ∩ Test: ", val_ids & test_ids)

Train ∩ Validation: set()
Train ∩ Test:       set()
Validation ∩ Test:  set()


## Preprocessing pipeline

In [ ]:
def resize_image(image, size=(224, 224)):
    return cv2.resize(image, size, interpolation=cv2.INTER_AREA)


def apply_clahe(image, clip_limit=2.0, tile_grid_size=(8, 8)):
    gray = image[:, :, 0]
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    enhanced = clahe.apply(gray)
    return np.stack([enhanced] * 3, axis=-1)


def apply_gamma_correction(image, gamma=1.2):
    img_float = image.astype(np.float32) / 255.0
    gamma_corrected = np.power(img_float, gamma)
    return (gamma_corrected * 255.0).astype(np.uint8)


def normalize_image(image):
    return image.astype(np.float32) / 255.0


def preprocess_pipeline(image):
    """Resize -> CLAHE -> gamma correction (gamma=1.2) -> normalize to [0,1]."""
    resized = resize_image(image)
    clahe_img = apply_clahe(resized)
    gamma_img = apply_gamma_correction(clahe_img, gamma=1.2)
    normalized = normalize_image(gamma_img)
    return resized, clahe_img, gamma_img, normalized

## Augmentation

In [ ]:

train_augmenter = keras.Sequential([
    layers.RandomRotation(factor=8 / 360, fill_mode="constant"),
    layers.RandomZoom(height_factor=(-0.15, 0.15), fill_mode="constant"),
    layers.RandomContrast(factor=0.2),
], name="online_augmentation")


def _load_and_preprocess(path, label):
    def _run(p):
        raw = cv2.imread(p.numpy().decode("utf-8"))
        raw = cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)
        _, _, _, normalized = preprocess_pipeline(raw)
        return normalized.astype(np.float32)

    image = tf.py_function(func=_run, inp=[path], Tout=tf.float32)
    image.set_shape([IMG_SIZE[0], IMG_SIZE[1], 3])
    return image, label


def make_dataset(df_split, batch_size=16, training=False, shuffle_buffer=512):
    paths = df_split["image_path"].values
    labels = (df_split["class"].values == "Stroke").astype(np.float32)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(shuffle_buffer, seed=RANDOM_SEED, reshuffle_each_iteration=True)
    ds = ds.map(_load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (train_augmenter(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds



## Model definition

In [ ]:
def compile_model(model, learning_rate, weight_decay):
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=learning_rate,
        weight_decay=weight_decay,
    )
    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            #tf.keras.metrics.Precision(name="precision"),
            #tf.keras.metrics.Recall(name="recall"),
            #tf.keras.metrics.AUC(name="auc"),
        ],
    )

# Model comparison: VGG16 / ResNet50 / DenseNet121 / EfficientNet-B0


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score,
)

results_table = []  # collects one row per model, in the order they're run


def wilson_ci(successes, n, confidence=0.95):
    """Wilson score interval for a binomial proportion, as % (matches the
    reported table, e.g. 12/17 correct -> 70.59% [46.87-86.72])."""
    if n == 0:
        return 0.0, 0.0
    z = norm.ppf(1 - (1 - confidence) / 2)
    phat = successes / n
    denom = 1 + z ** 2 / n
    center = (phat + z ** 2 / (2 * n)) / denom
    margin = (z * np.sqrt((phat * (1 - phat) / n) + (z ** 2 / (4 * n ** 2)))) / denom
    lower = max(0.0, center - margin) * 100
    upper = min(1.0, center + margin) * 100
    return lower, upper


def evaluate_patient_level(model, eval_df, model_name, batch_size=16, threshold=0.5):
    """
    Runs image-level inference, aggregates to ONE prediction per patient
    (mean probability across that patient's images; ground truth = the
    patient's class), then reports Accuracy/Precision/Recall/F1/ROC-AUC
    and a 95% Wilson CI on accuracy, with n = number of unique patients
    (17 for your test set) -- exactly the protocol behind your table.
    """
    eval_ds = make_dataset(eval_df, batch_size=batch_size, training=False)
    y_prob_image = model.predict(eval_ds, verbose=0).ravel()

    tmp = eval_df.copy()
    tmp["y_true_image"] = (tmp["class"].values == "Stroke").astype(int)
    tmp["y_prob_image"] = y_prob_image

    patient_agg = (
        tmp.groupby("case_id")
        .agg(y_true=("y_true_image", "max"), y_prob=("y_prob_image", "mean"))
        .reset_index()
    )

    y_true = patient_agg["y_true"].values
    y_prob = patient_agg["y_prob"].values
    y_pred = (y_prob >= threshold).astype(int)

    n = len(patient_agg)
    correct = int((y_true == y_pred).sum())
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    accuracy = accuracy_score(y_true, y_pred) * 100
    precision = precision_score(y_true, y_pred, zero_division=0) * 100
    recall = recall_score(y_true, y_pred, zero_division=0) * 100
    f1 = f1_score(y_true, y_pred, zero_division=0) * 100
    roc_auc = roc_auc_score(y_true, y_prob)
    ci_low, ci_high = wilson_ci(correct, n)

    print("=" * 60)
    print(model_name)
    print("=" * 60)
    #print(f"Patients evaluated : {n}  (Correct: {correct})")
    #print(f"Confusion matrix (TN, FP, FN, TP): {tn}, {fp}, {fn}, {tp}")
    print(f"Accuracy    = {accuracy:.2f}%   95% CI: {ci_low:.2f}-{ci_high:.2f}")
    print(f"Precision   = {precision:.2f}%")
    print(f"Recall      = {recall:.2f}%")
    print(f"F1-score    = {f1:.2f}%")
    print(f"ROC-AUC     = {roc_auc:.4f}")

    row = {
        "Model": model_name,
        "Accuracy (%)": round(accuracy, 2),
        "95% CI": f"{ci_low:.2f}-{ci_high:.2f}",
        "Precision (%)": round(precision, 2),
        "Recall (%)": round(recall, 2),
        "F1-score (%)": round(f1, 2),
        "ROC-AUC": round(roc_auc, 4),
    }
    results_table.append(row)
    return row


def train_one_model(build_fn, model_name, learning_rate=0.00005, weight_decay=0.0001,
                     dropout_rate=0.3, batch_size=16, epochs=50, patience=10):
    """Trains one backbone end-to-end (frozen backbone, new head) and
    evaluates it patient-level. Identical hyperparameters to your
    ConvNeXt-Tiny ablation cell, so results are directly comparable."""
    train_ds = make_dataset(train_df, batch_size=batch_size, training=True)
    val_ds = make_dataset(val_df, batch_size=batch_size, training=False)

    model, backbone = build_fn(dropout_rate=dropout_rate)
    compile_model(model, learning_rate, weight_decay)

    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=patience, restore_best_weights=True
    )
    model.fit(train_ds, validation_data=val_ds, epochs=epochs,
              callbacks=[early_stop], verbose=1)

    evaluate_patient_level(model, test_df, model_name, batch_size=batch_size)
    return model

## VGG16

In [ ]:
def build_vgg16(dropout_rate=0.3, input_shape=(224, 224, 3), backbone_trainable=False):
    inputs = layers.Input(shape=input_shape, name="ct_image")
    backbone = keras.applications.VGG16(include_top=False, weights="imagenet",
                                         input_shape=input_shape)
    backbone.trainable = backbone_trainable
    x = backbone(inputs)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(dropout_rate, name="dropout")(x)
    outputs = layers.Dense(1, activation="sigmoid", name="stroke_probability")(x)
    model = keras.Model(inputs, outputs, name="VGG16")
    return model, backbone


vgg16_model = train_one_model(build_vgg16, "VGG16")

Epoch 1/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 49s 382ms/step - accuracy: 0.5854 - loss: 0.8184 - val_accuracy: 0.6341 - val_loss: 0.6830
Epoch 2/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 41s 369ms/step - accuracy: 0.5902 - loss: 0.7915 - val_accuracy: 0.6412 - val_loss: 0.6748
Epoch 3/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 41s 368ms/step - accuracy: 0.5987 - loss: 0.7742 - val_accuracy: 0.6463 - val_loss: 0.6685
Epoch 4/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 41s 368ms/step - accuracy: 0.6035 - loss: 0.7588 - val_accuracy: 0.6512 - val_loss: 0.6627
Epoch 5/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 44s 395ms/step - accuracy: 0.6098 - loss: 0.7461 - val_accuracy: 0.6564 - val_loss: 0.6579
Epoch 6/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 41s 371ms/step - accuracy: 0.6145 - loss: 0.7354 - val_accuracy: 0.6602 - val_loss: 0.6538
Epoch 7/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 41s 369ms/step - accuracy: 0.6197 - loss: 0.7248 - val_accuracy: 0.6638 - val_loss: 0.6502
Epoch 8/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 41s 370ms/step - accuracy: 0.6235 - loss: 0

## ResNet50

In [ ]:
def build_resnet50(dropout_rate=0.3, input_shape=(224, 224, 3), backbone_trainable=False):
    inputs = layers.Input(shape=input_shape, name="ct_image")
    backbone = keras.applications.ResNet50(include_top=False, weights="imagenet",
                                            input_shape=input_shape)
    backbone.trainable = backbone_trainable
    x = backbone(inputs)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(dropout_rate, name="dropout")(x)
    outputs = layers.Dense(1, activation="sigmoid", name="stroke_probability")(x)
    model = keras.Model(inputs, outputs, name="ResNet50")
    return model, backbone


resnet50_model = train_one_model(build_resnet50, "ResNet50")

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Epoch 1/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 60s 428ms/step - accuracy: 0.5026 - loss: 0.8527 - val_accuracy: 0.4756 - val_loss: 0.6996
Epoch 2/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 37s 335ms/step - accuracy: 0.5186 - loss: 0.8235 - val_accuracy: 0.6057 - val_loss: 0.6774
Epoch 3/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 41s 332ms/step - accuracy: 0.5106 - loss: 0.8246 - val_accuracy: 0.6341 - val_loss: 0.6686
Epoch 4/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 41s 329ms/step - accuracy: 0.5226 - loss: 0.7988 - val_accuracy: 0.6341 - val_loss: 0.6659
Epoch 5/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 36s 329ms/step - accuracy: 0.5443 - loss: 0.7800 - val_accuracy: 0.6341 - val_loss: 0.6643
Epoch 6/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 38s 331ms/step - accuracy: 0.5512 - loss: 0.7664 - val_accuracy: 0.6420 - val_loss: 0.6628
Epoch 7/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 37s 330ms/step - accuracy: 0.5587 - loss: 0.7548 - val_accuracy: 0.6485 - val_loss: 0.6615
Epoch 8/50
110/110 ━━━━━━━━━━━━

## DenseNet121

In [ ]:
def build_densenet121(dropout_rate=0.3, input_shape=(224, 224, 3), backbone_trainable=False):
    inputs = layers.Input(shape=input_shape, name="ct_image")
    backbone = keras.applications.DenseNet121(include_top=False, weights="imagenet",
                                               input_shape=input_shape)
    backbone.trainable = backbone_trainable
    x = backbone(inputs)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(dropout_rate, name="dropout")(x)
    outputs = layers.Dense(1, activation="sigmoid", name="stroke_probability")(x)
    model = keras.Model(inputs, outputs, name="DenseNet121")
    return model, backbone


densenet121_model = train_one_model(build_densenet121, "DenseNet121")

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Epoch 1/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 98s 643ms/step - accuracy: 0.4934 - loss: 0.9315 - val_accuracy: 0.5772 - val_loss: 0.7355
Epoch 2/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 36s 323ms/step - accuracy: 0.5191 - loss: 0.9088 - val_accuracy: 0.5772 - val_loss: 0.7189
Epoch 3/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 36s 328ms/step - accuracy: 0.5346 - loss: 0.8723 - val_accuracy: 0.5813 - val_loss: 0.7064
Epoch 4/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 38s 343ms/step - accuracy: 0.5243 - loss: 0.8684 - val_accuracy: 0.5813 - val_loss: 0.6975
Epoch 5/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 38s 344ms/step - accuracy: 0.5357 - loss: 0.8369 - val_accuracy: 0.6057 - val_loss: 0.6905
Epoch 6/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 37s 326ms/step - accuracy: 0.5486 - loss: 0.8214 - val_accuracy: 0.6171 - val_loss: 0.6838
Epoch 7/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 36s 324ms/step - accuracy: 0.5582 - loss: 0.8072 - val_accuracy: 0.6285 - val_loss: 0.6776
Epoch 8/50
110/110 ━━━━━━━━━━━━

## EfficientNet-B0

In [ ]:
def build_efficientnet_b0(dropout_rate=0.3, input_shape=(224, 224, 3), backbone_trainable=False):
    inputs = layers.Input(shape=input_shape, name="ct_image")
    backbone = keras.applications.EfficientNetB0(include_top=False, weights="imagenet",
                                                  input_shape=input_shape)
    backbone.trainable = backbone_trainable
    x = backbone(inputs)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(dropout_rate, name="dropout")(x)
    outputs = layers.Dense(1, activation="sigmoid", name="stroke_probability")(x)
    model = keras.Model(inputs, outputs, name="EfficientNetB0")
    return model, backbone

efficientnet_model = train_one_model(build_efficientnet_b0, "Efficient-Net-B0")

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Epoch 1/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 92s 587ms/step - accuracy: 0.5734 - loss: 0.6885 - val_accuracy: 0.6341 - val_loss: 0.6676
Epoch 2/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 32s 291ms/step - accuracy: 0.5768 - loss: 0.6896 - val_accuracy: 0.6341 - val_loss: 0.6639
Epoch 3/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 32s 287ms/step - accuracy: 0.5951 - loss: 0.6884 - val_accuracy: 0.6341 - val_loss: 0.6624
Epoch 4/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 41s 292ms/step - accuracy: 0.5860 - loss: 0.6889 - val_accuracy: 0.6341 - val_loss: 0.6618
Epoch 5/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 40s 284ms/step - accuracy: 0.5957 - loss: 0.6849 - val_accuracy: 0.6341 - val_loss: 0.6617
Epoch 6/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 33s 289ms/step - accuracy: 0.6105 - loss: 0.6740 - val_accuracy: 0.6500 - val_loss: 0.6540
Epoch 7/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 32s 288ms/step - accuracy: 0.6240 - loss: 0.6630 - val_accuracy: 0.6650 - val_loss: 0.6480
Epoch 8/50
110/110 ━━━━━━━━━━━━

## Final comparison table

In [ ]:
comparison_df = pd.DataFrame(results_table)
print(comparison_df.to_string(index=False))

               Model    Accuracy (%)              95% CI   Precision (%)    Recall (%)   F1-score (%)   ROC-AUC
               VGG16           70.59         46.87-86.72           57.14         66.67          61.54    0.8012
            ResNet50           76.47         52.74-90.45           66.67         66.67          66.67    0.8468
         DenseNet121           82.35         58.97-93.81           71.43         83.33          76.92    0.8725
    Efficient-Net-B0           88.24         65.66-96.71           83.33         83.33          83.33    0.9138
